In [ ]:
# Parameters — modify these to analyze a different model
MODEL_ID  = "lgbm_solusdt_l_fw60_2101_2605"
DIRECTION = "long"   # "long" or "short"
TARGET    = "long_mfe_fw60"   # set by DIRECTION: long->long_mfe_fw60ng_mfe_fw60, s->long_mfe_fw60


In [ ]:
import sys
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
from IPython.display import display, Markdown

from analyst.lib.db_utils import find_repo_root, lab_db_path, db_path as _live_db_path
from analyst.lib.plot_utils import CQ_COLORS, CQ_SEQUENCE, setup_cq_theme
from analyst.lib.table_formatting import format_analysis_table, display_analysis_table

_root = find_repo_root()
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))
setup_cq_theme()

# --- Adatbetöltés ---
LAB_DB = lab_db_path()
LIVE_DB = _live_db_path()

con = duckdb.connect(LAB_DB, read_only=True)
df = con.execute(
    f'SELECT open_time, {TARGET}, split FROM model."{MODEL_ID}__sample"'
).fetchdf()
con.close()

df["open_time"] = pd.to_datetime(df["open_time"])
df["segment"] = df["split"].map({0: "Train", 1: "Valid"})

VALID_START = df[df["segment"] == "Valid"]["open_time"].min()


## Cél

Ez a notebook a `lgbm_solusdt_l_fw60_2101_2605` short champion modell **sampling periódusát** vizsgálja.
Vizsgált területek: train/valid szétválasztás, megfigyelésszámok, dátumtartományok, átlagos target
(`long_mfe_fw60`) időbeli alakulása, és a target eloszlásának stabilitása train vs valid között.

**Kapcsolódó dokumentáció:**
- Módszertan: `_doc_/methodology_doc/3000_sampling.md`
- Kód-dokumentáció: `_doc_/database_and_code_doc/5300_create_sample.md`

**Asset:** SOLUSDT | **Granularitás:** 1m (óránkénti mintavétel) | **Dátum:** 2026-06-29

## 1. Train / Valid összesítő

**Mi ez.** Az összes sample-sor szétválasztva train (split=0) és valid (split=1) szerint.

**Forrás.** `model."lgbm_solusdt_l_fw60_2101_2605__sample"` tábla (`open_time`, `long_mfe_fw60`, `split`).

**Módszer.** Csoportosítás `split` szerint: megfigyelések száma, első és utolsó időpont, átlagos
`long_mfe_fw60` és negatív arány (a target < 0 hányada — negatív short_mfe = nyereséges short).

**Értelmezés.** Megmutatja, arányos-e a train/valid felosztás, és van-e eltérés az átlagos target
szintjében a két periódus között.

In [ ]:
#| label: tbl-train-valid-summary
#| tbl-cap: "Train és Valid összesítő — megfigyelésszám, időtartam, átlagos long_mfe_fw60, negatív arány"

summary = (
    df.groupby("segment")
    .agg(
        n_obs=("open_time", "count"),
        min_date=("open_time", "min"),
        max_date=("open_time", "max"),
        avg_target=(TARGET, "mean"),
        negative_rate=(TARGET, lambda x: (x < 0).mean()),
    )
    .reset_index()
)
summary["min_date"] = summary["min_date"].dt.strftime("%Y-%m-%d")
summary["max_date"] = summary["max_date"].dt.strftime("%Y-%m-%d")
summary = summary.rename(columns={
    "segment": "Szegmens",
    "n_obs": "Megfigyelések (db)",
    "min_date": "Első adat",
    "max_date": "Utolsó adat",
    "avg_target": "Átlag long_mfe_fw60",
    "negative_rate": "Negatív arány (short nyereség)",
})
display_analysis_table(summary)


## 2. Havi target alakulása

**Mi ez.** A `long_mfe_fw60` havi átlaga és szórássávja a teljes mintavételi perióduson.

**Forrás.** `model."lgbm_solusdt_l_fw60_2101_2605__sample"` tábla, havi aggregáció.

**Módszer.** Havi bontású csoportosítás: átlag és szórás kiszámítása `long_mfe_fw60`-ra.
A vonal az átlagot, a sáv (±1σ) a szórást mutatja. A narancs terület jelöli a valid időszakot
(2025-05 –), a szaggatott vonal a train/valid határt.

**Értelmezés.** Megmutatja, hogy a target átlagos szintje stabil-e az időben, és van-e
rezsimváltás a train és valid periódus között. Negatív értékek kedvezők (profitábilis short irány).

In [ ]:
#| label: fig-monthly-target
#| fig-cap: "Havi átlagos long_mfe_fw60 és ±1σ szórássáv — train (kék) / valid (narancs terület)"
#| fig-alt: "Vonalas ábra, x: hónap (2021-01–2026-05), y: átlag long_mfe_fw60. Kék sáv = ±1σ. Narancs terület = valid időszak. Szaggatott vonal = train/valid határ."

df["month"] = df["open_time"].dt.to_period("M")
monthly = (
    df.groupby("month")
    .agg(avg_target=(TARGET, "mean"), std_target=(TARGET, "std"))
    .reset_index()
)
monthly["date"] = monthly["month"].dt.to_timestamp()
monthly["upper"] = monthly["avg_target"] + monthly["std_target"]
monthly["lower"] = monthly["avg_target"] - monthly["std_target"]

valid_start_month = VALID_START.replace(day=1)
plot_end = monthly["date"].max() + pd.DateOffset(months=1)

fig, ax = plt.subplots(figsize=(10, 5))

ax.fill_between(
    monthly["date"], monthly["lower"], monthly["upper"],
    alpha=0.18, color=CQ_COLORS["blue"], label="±1σ sáv"
)
ax.plot(
    monthly["date"], monthly["avg_target"],
    color=CQ_COLORS["blue"], linewidth=1.6, label="Átlag long_mfe_fw60"
)
ax.axvspan(
    valid_start_month, plot_end,
    alpha=0.08, color=CQ_COLORS["orange"], label="Valid időszak"
)
ax.axvline(
    valid_start_month, color=CQ_COLORS["orange"],
    linewidth=1.3, linestyle="--",
    label=f"Valid kezdet: {VALID_START.strftime('%Y-%m')}"
)
ax.axhline(0, color=CQ_COLORS["gray"], linewidth=0.8, linestyle=":")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
plt.xticks(rotation=45, ha="right")
ax.set_xlabel("Hónap")
ax.set_ylabel("long_mfe_fw60 (log return)")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()


## 3. Target eloszlása (Train vs Valid)

**Mi ez.** A `long_mfe_fw60` eloszlásának összehasonlítása train és valid periódus között.

**Forrás.** `model."lgbm_solusdt_l_fw60_2101_2605__sample"`, `split` szegmens szerint szétválasztva.

**Módszer.** (a) KDE sűrűségbecslés: train és valid eloszlás egymáson; (b) boxplot: train és
valid egymás mellett — medián, IQR és outlier-ek vizualizálva.

**Értelmezés.** Az eloszlások hasonlósága jelzi, hogy a valid periódus reprezentatív-e
a train mintához képest. Jelentős eltolódás rezsimváltást vagy distribution shift-et jelez.
Short modellnél a negatív oldal a profitábilis rész.

In [ ]:
#| label: fig-distribution-kde
#| fig-cap: "long_mfe_fw60 eloszlása — Train (kék) és Valid (narancs) KDE egymáson"
#| fig-alt: "KDE sűrűségábra. Train eloszlás kék, valid narancs, egymáson ábrázolva. Szaggatott 0 vonal."

train_data = df[df["segment"] == "Train"][TARGET]
valid_data = df[df["segment"] == "Valid"][TARGET]

fig, ax = plt.subplots(figsize=(9, 5))
sns.kdeplot(
    train_data, ax=ax,
    color=CQ_COLORS["blue"], label="Train",
    linewidth=1.8, fill=True, alpha=0.12
)
sns.kdeplot(
    valid_data, ax=ax,
    color=CQ_COLORS["orange"], label="Valid",
    linewidth=1.8, fill=True, alpha=0.12
)
ax.axvline(0, color=CQ_COLORS["gray"], linewidth=0.9, linestyle="--", label="0 vonal")
ax.set_xlabel("long_mfe_fw60 (log return)")
ax.set_ylabel("Sűrűség")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
#| label: fig-distribution-boxplot
#| fig-cap: "long_mfe_fw60 boxplot — Train és Valid egymás mellett"
#| fig-alt: "Boxplot. Train (kék) és Valid (narancs) egymás mellett. Medián, IQR és outlier-ek jelölve."

segments = ["Train", "Valid"]
colors_bp = [CQ_COLORS["blue"], CQ_COLORS["orange"]]
data_by_seg = [train_data.values, valid_data.values]

fig, ax = plt.subplots(figsize=(7, 5))
bp = ax.boxplot(
    data_by_seg,
    labels=segments,
    patch_artist=True,
    medianprops=dict(color=CQ_COLORS["black"], linewidth=1.8),
    whiskerprops=dict(linewidth=1.0, color=CQ_COLORS["gray_dark"]),
    capprops=dict(linewidth=1.0, color=CQ_COLORS["gray_dark"]),
    flierprops=dict(marker="o", markersize=2, alpha=0.25, color=CQ_COLORS["gray"]),
    showfliers=True,
)
for patch, color in zip(bp["boxes"], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.35)

ax.axhline(0, color=CQ_COLORS["gray"], linewidth=0.9, linestyle="--")
ax.set_ylabel("long_mfe_fw60 (log return)")
plt.tight_layout()
plt.show()


## Értelmezés

In [ ]:
#| label: tbl-interp-metrics
#| tbl-cap: "Összehasonlító metrikák — Train vs Valid"

rows = []
for seg, data in [("Train", train_data), ("Valid", valid_data)]:
    rows.append({
        "Szegmens": seg,
        "Medián": data.median(),
        "Átlag": data.mean(),
        "Szórás (σ)": data.std(),
        "IQR": data.quantile(0.75) - data.quantile(0.25),
        "Negatív arány (short nyereség)": (data < 0).mean(),
    })
interp_df = pd.DataFrame(rows)
display_analysis_table(interp_df)


## 4. Top10 decilis havi bontásban — Train és Valid

**Struktúra.** Train és Valid hónapjai egyetlen táblában, éves összesítő sorokkal.
Short modellnél a **top10 = a legalacsonyabb (leg-negatívabb) értékek** — P10 küszöb szegmensenkénti.
A top10 küszöböt szegmensenkéből külön számoljuk: **P10 (train)** és **P10 (valid)**.

**Tábla.** Szegmens / Időszak / N sample / N top10 / Top10 avg target / Top10 / hét / Top10 / nap —
az éves összesítő sorok dőlten szedve.

**Ábra.** Havi "top10 / nap" trend — Train (kék) és Valid (narancs) külön sorozatként,
az éves összesítőkből számítva. Narancs terület jelöli a valid periódust.

In [ ]:
#| label: tbl-top10-summary
#| tbl-cap: "Top10 decilis összehasonlítás — Train vs Valid (szegmensenkénti P10, short: legnegatívabb értékek)"

train_df = df[df["segment"] == "Train"][["open_time", TARGET]].copy()
valid_df = df[df["segment"] == "Valid"][["open_time", TARGET]].copy()

# Short esetén a top10 = a legalacsonyabb (leg-negatívabb) értékek => P10
p10_train = np.percentile(train_df[TARGET].dropna(), 10)
p10_valid = np.percentile(valid_df[TARGET].dropna(), 10)

t10_train_avg = train_df[train_df[TARGET] <= p10_train][TARGET].mean()
t10_valid_avg = valid_df[valid_df[TARGET] <= p10_valid][TARGET].mean()
global_avg    = valid_df[TARGET].mean()

# P10 + top10 avg referenciatábla
ref_df = pd.DataFrame([
    {"Metrika": "P10 küszöb (szegmensenkénti — short: legnegatívabb 10%)",  "Train": f"{p10_train:.5f}", "Valid": f"{p10_valid:.5f}"},
    {"Metrika": "Top10 avg target (short: legnegatívabb értékek átlaga)",   "Train": f"{t10_train_avg:.5f}", "Valid": f"{t10_valid_avg:.5f}"},
    {"Metrika": "Teljes szegmens avg target",                                "Train": f"{train_df[TARGET].mean():.5f}", "Valid": f"{valid_df[TARGET].mean():.5f}"},
    {"Metrika": "Lift (|top10_avg| / |segment_avg|)",                        "Train": f"{abs(t10_train_avg) / abs(train_df[TARGET].mean()):.3f}x", "Valid": f"{abs(t10_valid_avg) / abs(valid_df[TARGET].mean()):.3f}x"},
])
display_analysis_table(ref_df)

# Heti grid a teljes adatperióduson
all_start = df["open_time"].min().normalize()
all_end   = df["open_time"].max().normalize() + pd.Timedelta(days=1)
week_starts_all = pd.date_range(start=all_start, end=all_end, freq="7D")

# Havi + éves aggregátum build
monthly_rows = []
for seg_label, seg_df, p10 in [("Train", train_df, p10_train), ("Valid", valid_df, p10_valid)]:
    sd = seg_df.copy()
    sd["top10"] = sd[TARGET] <= p10  # short: leg-negatívabb értékek
    sd["ym"]    = sd["open_time"].dt.to_period("M")
    sd["year"]  = sd["open_time"].dt.year

    for year, ydf in sd.groupby("year"):
        for ym, mdf in ydf.groupby("ym"):
            n_s = len(mdf)
            n_t = int(mdf["top10"].sum())
            n_d = ym.days_in_month
            n_w = sum(1 for ws in week_starts_all if ws.strftime("%Y-%m") == str(ym))
            t_avg = mdf[mdf["top10"]][TARGET].mean() if n_t else np.nan
            monthly_rows.append({
                "_order": 0,
                "Szegmens": seg_label,
                "Időszak":  str(ym),
                "N sample": n_s,
                "N top10":  n_t,
                "Top10 avg target (short: legnegatívabb)": round(t_avg, 5) if not np.isnan(t_avg) else None,
                "Top10 / hét":      round(n_t / n_w, 1) if n_w else None,
                "Top10 / nap":      round(n_t / n_d, 2),
            })

        # Éves összesítő
        n_sy = len(ydf); n_ty = int(ydf["top10"].sum())
        n_dy = (ydf["open_time"].dt.date.max() - ydf["open_time"].dt.date.min()).days + 1
        t_avg_y = ydf[ydf["top10"]][TARGET].mean() if n_ty else np.nan
        monthly_rows.append({
            "_order": 1,
            "Szegmens": f"{seg_label} {year} ▶",
            "Időszak":  f"{year} összesen",
            "N sample": n_sy,
            "N top10":  n_ty,
            "Top10 avg target (short: legnegatívabb)": round(t_avg_y, 5) if not np.isnan(t_avg_y) else None,
            "Top10 / hét":      None,
            "Top10 / nap":      round(n_ty / n_dy, 2),
        })

combined_df = pd.DataFrame(monthly_rows).drop(columns=["_order"])
display_analysis_table(combined_df)


In [ ]:
#| label: fig-top10-monthly-trend
#| fig-cap: "Havi átlagos top10 decilis / nap — Train (kék) és Valid (narancs), szegmensenkénti P10 (short: legnegatívabb 10%)"
#| fig-height: 5

# Havi plot adatok
plot_rows = []
for seg_label, seg_df, p10 in [("Train", train_df, p10_train), ("Valid", valid_df, p10_valid)]:
    sd = seg_df.copy()
    sd["top10"] = sd[TARGET] <= p10  # short: leg-negatívabb értékek
    sd["ym"]    = sd["open_time"].dt.to_period("M")
    for ym, mdf in sd.groupby("ym"):
        n_t = int(mdf["top10"].sum())
        plot_rows.append({
            "segment":      seg_label,
            "date":         ym.to_timestamp(),
            "top10_per_day": n_t / ym.days_in_month,
        })

plot_df      = pd.DataFrame(plot_rows)
train_plot   = plot_df[plot_df["segment"] == "Train"]
valid_plot   = plot_df[plot_df["segment"] == "Valid"]

valid_start_dt = valid_df["open_time"].min().replace(day=1)
plot_end_dt    = valid_df["open_time"].max().replace(day=1) + pd.DateOffset(months=1)

fig, ax = plt.subplots(figsize=(12, 5))

ax.axvspan(valid_start_dt, plot_end_dt,
           alpha=0.08, color=CQ_COLORS["orange"], label="Valid időszak")
ax.axvline(valid_start_dt, color=CQ_COLORS["orange"], linewidth=1.3, linestyle="--",
           label=f"Valid kezdet ({valid_start_dt.strftime('%Y-%m')})")

ax.plot(train_plot["date"], train_plot["top10_per_day"],
        color=CQ_COLORS["blue"], linewidth=1.4, marker="o", markersize=3,
        label=f"Train top10/nap  (P10={p10_train:.5f})")
ax.plot(valid_plot["date"], valid_plot["top10_per_day"],
        color=CQ_COLORS["orange"], linewidth=1.6, marker="o", markersize=4,
        label=f"Valid top10/nap  (P10={p10_valid:.5f})")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
plt.xticks(rotation=45, ha="right")
ax.set_xlabel("Hónap")
ax.set_ylabel("Top10 decilis / nap (db)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 5. Top 10 lift ábrák — Train és Valid

Vizuális áttekintés arról, hogy **mikor és milyen close árfolyam mellett** jelennek meg a top 10%
decilisbe eső target (`long_mfe_fw60`) megfigyelések.
**Short modellnél: top10 = a legalacsonyabb (leg-negatívabb) értékek (P10 küszöb alattiak).**
A piros szaggatott függőleges vonalak mindkét panelen jelzik a top decilis x-pozícióit.

- **Train:** éves alfejezetek, minden évben havi tabset (1 fül = 1 naptári hónap). P10 küszöb:
  a teljes train halmaz 10. percentilise.
- **Valid:** havi alfejezetek, minden hónapban heti tabset (1 fül = 7 nap). P10 küszöb:
  a teljes valid halmaz 10. percentilise.

### Train — éves bontás, havi tabset

In [ ]:
#| label: fig-train-yearly-tabsets
#| output: asis
#| warning: false
#| echo: false

import base64, io, calendar

TOP_DECILE_COLOR = "#d62728"
TABSET_OPEN  = "::: {.panel-tabset}"
TABSET_CLOSE = ":::"

# p10_train, train_df definiálva a tbl-top10-summary cellában
train_start = train_df["open_time"].min().normalize()
train_end   = train_df["open_time"].max().normalize() + pd.Timedelta(days=1)
train_avg   = train_df[TARGET].mean()

con_live = duckdb.connect(LIVE_DB, read_only=True)
ohlcv_train = con_live.execute(
    f"SELECT open_time, close FROM ohlcv "
    f"WHERE open_time >= '{train_start}' AND open_time < '{train_end}' "
    f"ORDER BY open_time"
).fetchdf()
con_live.close()
ohlcv_train["open_time"] = pd.to_datetime(ohlcv_train["open_time"])

train_full2 = train_df.copy()
# Short: top decilis = leg-negatívabb értékek (target <= p10_train)
train_full2["top_decile"] = train_full2[TARGET] <= p10_train
train_full2["ym"]   = train_full2["open_time"].dt.to_period("M")
train_full2["year"] = train_full2["open_time"].dt.year

for year, ydf in train_full2.groupby("year"):
    print(f"\n#### {year}\n")
    print(TABSET_OPEN + "\n")

    for ym, mdf in ydf.groupby("ym"):
        m_start  = ym.to_timestamp()
        m_end    = (ym + 1).to_timestamp()
        ohlcv_m  = ohlcv_train[(ohlcv_train["open_time"] >= m_start) & (ohlcv_train["open_time"] < m_end)]
        top_rows  = mdf[mdf["top_decile"]]
        top_times = top_rows["open_time"].tolist()

        print(f"## {calendar.month_abbr[ym.month]}\n")

        fig, (ax1, ax2) = plt.subplots(
            2, 1, sharex=True, figsize=(13, 6),
            gridspec_kw={"height_ratios": [2, 1], "hspace": 0.05}
        )

        for t in top_times:
            ax1.axvline(t, color=TOP_DECILE_COLOR, linewidth=0.8, linestyle="--", alpha=0.6, zorder=1)
            ax2.axvline(t, color=TOP_DECILE_COLOR, linewidth=0.8, linestyle="--", alpha=0.6, zorder=1)

        if len(ohlcv_m):
            ax1.plot(ohlcv_m["open_time"], ohlcv_m["close"],
                     color=CQ_COLORS["blue"], linewidth=0.6, zorder=2)
        ax1.set_ylabel("Close (USDT)")
        ax1.tick_params(axis="x", labelbottom=False)

        if len(mdf):
            ax2.plot(mdf["open_time"], mdf[TARGET],
                     color=CQ_COLORS["black"], linewidth=1.0,
                     marker="o", markersize=1.5, zorder=3)
        if len(top_rows):
            ax2.scatter(top_rows["open_time"], top_rows[TARGET],
                        color=TOP_DECILE_COLOR, s=25, zorder=4)
        ax2.axhline(p10_train, color=CQ_COLORS["gray"], linewidth=0.9, linestyle=":",
                    label=f"P10 train ({p10_train:.5f})", zorder=3)
        ax2.set_ylabel("long_mfe_fw60")
        ax2.set_xlabel("Dátum (UTC)")
        ax2.legend(loc="upper left", fontsize=8)

        ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
        ax2.xaxis.set_major_locator(mdates.DayLocator(interval=5))
        plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

        n_t = len(top_rows)
        t_avg = top_rows[TARGET].mean() if n_t else np.nan
        # Short lift: |top10_avg| / |train_avg|
        lift  = abs(t_avg) / abs(train_avg) if (n_t and train_avg != 0) else np.nan
        lift_str = f"lift={lift:.2f}x" if not np.isnan(lift) else "lift=n/a"
        fig.suptitle(
            f"{ym}  |  n={len(mdf)}  top10={n_t}  {lift_str}  P10={p10_train:.5f}",
            fontsize=9, y=1.01
        )
        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=85, bbox_inches="tight")
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode()
        plt.close(fig)

        print(f"![](data:image/png;base64,{img_b64})\n")

    print(TABSET_CLOSE + "\n")


### Valid — havi bontás, heti tabset

In [ ]:
#| label: fig-valid-monthly-tabsets
#| output: asis
#| warning: false
#| echo: false

import base64, io, calendar

TOP_DECILE_COLOR = "#d62728"
TABSET_OPEN  = "::: {.panel-tabset}"
TABSET_CLOSE = ":::"

valid_full  = valid_df.copy()
p10_global  = p10_valid
valid_start = valid_df["open_time"].min().normalize()
valid_end   = valid_df["open_time"].max().normalize() + pd.Timedelta(days=1)
week_starts = pd.date_range(start=valid_start, end=valid_end, freq="7D")

con_live = duckdb.connect(LIVE_DB, read_only=True)
ohlcv_valid = con_live.execute(
    f"SELECT open_time, close FROM ohlcv "
    f"WHERE open_time >= '{valid_start}' AND open_time < '{valid_end}' "
    f"ORDER BY open_time"
).fetchdf()
con_live.close()
ohlcv_valid["open_time"] = pd.to_datetime(ohlcv_valid["open_time"])

valid_full2 = valid_full.copy()
# Short: top decilis = leg-negatívabb értékek (target <= p10_global)
valid_full2["top_decile"] = valid_full2[TARGET] <= p10_global

weeks_by_month = {}
for ws in week_starts:
    we = ws + pd.Timedelta(days=7)
    sw = valid_full2[(valid_full2["open_time"] >= ws) & (valid_full2["open_time"] < we)]
    if len(sw) == 0:
        continue
    mk = ws.strftime("%Y-%m")
    weeks_by_month.setdefault(mk, []).append((ws, we))

for month_str, week_list in weeks_by_month.items():
    y, m = int(month_str[:4]), int(month_str[5:])
    month_name = f"{calendar.month_abbr[m]} {y}"
    print(f"\n#### {month_name}\n")
    print(TABSET_OPEN + "\n")

    for ws, we in week_list:
        ohlcv_w  = ohlcv_valid[(ohlcv_valid["open_time"] >= ws) & (ohlcv_valid["open_time"] < we)]
        sample_w = valid_full2[(valid_full2["open_time"] >= ws) & (valid_full2["open_time"] < we)].copy()
        top_rows  = sample_w[sample_w["top_decile"]]
        top_times = top_rows["open_time"].tolist()

        tab_label = f"{ws.strftime('%m-%d')}–{(we - pd.Timedelta(days=1)).strftime('%m-%d')}"
        print(f"## {tab_label}\n")

        fig, (ax1, ax2) = plt.subplots(
            2, 1, sharex=True, figsize=(13, 6),
            gridspec_kw={"height_ratios": [2, 1], "hspace": 0.05}
        )

        for t in top_times:
            ax1.axvline(t, color=TOP_DECILE_COLOR, linewidth=1.0, linestyle="--", alpha=0.7, zorder=1)
            ax2.axvline(t, color=TOP_DECILE_COLOR, linewidth=1.0, linestyle="--", alpha=0.7, zorder=1)

        if len(ohlcv_w):
            ax1.plot(ohlcv_w["open_time"], ohlcv_w["close"],
                     color=CQ_COLORS["blue"], linewidth=0.9, label="Close (USDT)", zorder=2)
        ax1.set_ylabel("Close (USDT)")
        ax1.tick_params(axis="x", labelbottom=False)
        ax1.legend(loc="upper left", fontsize=8)

        if len(sample_w):
            ax2.plot(sample_w["open_time"], sample_w[TARGET],
                     color=CQ_COLORS["black"], linewidth=1.2, marker="o", markersize=2,
                     label="long_mfe_fw60", zorder=3)
        if len(top_rows):
            ax2.scatter(top_rows["open_time"], top_rows[TARGET],
                        color=TOP_DECILE_COLOR, s=35, zorder=4, label="Top 10% decilis (legnegatívabb)")
        ax2.axhline(p10_global, color=CQ_COLORS["gray"], linewidth=0.9, linestyle=":",
                    label=f"P10 valid ({p10_global:.5f})", zorder=3)
        ax2.set_ylabel("long_mfe_fw60")
        ax2.set_xlabel("Dátum (UTC)")
        ax2.legend(loc="upper left", fontsize=8)

        ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
        ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))
        plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

        top10_avg_w = top_rows[TARGET].mean() if len(top_rows) else float("nan")
        lift_w = abs(top10_avg_w) / abs(global_avg) if (len(top_rows) and global_avg != 0) else float("nan")
        lift_str = f"lift={lift_w:.2f}x" if not np.isnan(lift_w) else "lift=n/a"
        fig.suptitle(
            f"{ws.strftime('%Y-%m-%d')} – {(we - pd.Timedelta(days=1)).strftime('%Y-%m-%d')}"
            f"  |  n={len(sample_w)}  top10={len(top_rows)}  {lift_str}  P10={p10_global:.5f}",
            fontsize=9, y=1.01
        )
        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode()
        plt.close(fig)

        print(f"![](data:image/png;base64,{img_b64})\n")

    print(TABSET_CLOSE + "\n")
